# E_S2 — Full metrics computation for the regenerated SNR sweep

This notebook has one responsibility: load the cases generated by `D_S3_snr_sweep_generation.ipynb`, compute the same full metric set as `D_S2_metrics_computation.ipynb`, and save the case-level metric table.

It does **not** aggregate MIC, connect human labels, calibrate labels, or create figures. The full run is saved in resumable Parquet chunks and consolidated into `metrics_full.parquet` with the same 77-column schema as the original D_S2 output.

In [5]:
from __future__ import annotations

import json
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.spatial.distance import pdist, squareform
from scipy.stats import ks_2samp, pearsonr, wasserstein_distance
from statsmodels.nonparametric.smoothers_lowess import lowess

try:
    from minepy import MINE
    HAS_MINEPY = True
except ImportError:
    HAS_MINEPY = False
    print('minepy is unavailable — MIC/MAS/MEV/MCN would be NaN.')

try:
    from pygam import LinearGAM, s as gam_s
    HAS_PYGAM = True
except ImportError:
    HAS_PYGAM = False
    print('pygam is unavailable — GAM metrics would be NaN.')

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kwargs):
        return it

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 180)

## Load and validate the generated cases

In [6]:
RUN_MODE = os.environ.get('SNR_SWEEP_MODE', 'full').strip().lower()
if RUN_MODE not in {'full', 'pilot'}:
    raise ValueError("SNR_SWEEP_MODE must be 'full' or 'pilot'")

RESUME = os.environ.get('SNR_METRICS_RESUME', '1') == '1'
OVERWRITE_FINAL = os.environ.get('SNR_METRICS_OVERWRITE', '0') == '1'
CHUNK_SIZE = int(os.environ.get('SNR_METRICS_CHUNK_SIZE', '500' if RUN_MODE == 'full' else '20'))

def locate_repo_root() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in candidates:
        if (root / 'D' / 'Global').is_dir():
            return root
    raise FileNotFoundError('Could not locate D/D_S2_metrics_computation.ipynb')

REPO_ROOT = locate_repo_root()
GLOBAL_DIR = REPO_ROOT / 'D' / 'Global'
BASE_OUTPUT_DIR = GLOBAL_DIR / 'output' / 'S3_snr_mic_sweep'
DATA_DIR = BASE_OUTPUT_DIR if RUN_MODE == 'full' else BASE_OUTPUT_DIR / '_pilot'
CHUNK_DIR = DATA_DIR / 'metric_chunks'
METRIC_DEFINITION_SOURCE = 'Embedded snapshot copied from D/D_S2_metrics_computation.ipynb'
REFERENCE_METRICS = REPO_ROOT / 'D' / 'output' / 'S2' / 'metrics_full.parquet'

required = [DATA_DIR / 'cases.csv', DATA_DIR / 'scatter_points.npz', DATA_DIR / 'generation_metadata.json']
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Run D_S3 first. Missing: ' + ', '.join(map(str, missing)))

cases_df = pd.read_csv(DATA_DIR / 'cases.csv', low_memory=False)
points = np.load(DATA_DIR / 'scatter_points.npz')
x_all = points['x']
y_all = points['y']
n_cases, n_points = x_all.shape

assert x_all.shape == y_all.shape
assert len(cases_df) == n_cases
assert cases_df['candidate_index'].to_numpy().tolist() == list(range(n_cases))
assert cases_df['case_id'].is_unique
if not HAS_MINEPY:
    raise ImportError('minepy is required. Run this notebook with the pip39 kernel.')

CHUNK_DIR.mkdir(parents=True, exist_ok=True)
print(f'RUN_MODE: {RUN_MODE}')
print(f'Data: {DATA_DIR}')
print(f'Cases: {n_cases:,}; points/case: {n_points}')
print(f'Chunk size: {CHUNK_SIZE}; resume: {RESUME}')
display(cases_df.groupby(['family_id', 'variant_level'], dropna=False).size().rename('n').reset_index())

RUN_MODE: full
Data: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S3_snr_mic_sweep
Cases: 95,440; points/case: 500
Chunk size: 500; resume: True


,family_id,variant_level,n
0,F01,mild,2270
1,F01,standard,2270
2,F01,strong,2270
3,F03,mild,2270
4,F03,standard,2270
5,F03,strong,2270
6,F05,mild,2270
7,F05,standard,2270
8,F05,strong,2270
9,F07,mild,2270


## Embedded snapshot of the complete D_S2 metric definitions

The following functions are copied into this notebook from `D_S2_metrics_computation.ipynb`, so this run does not read or execute another notebook. The snapshot includes vectorised Pearson/Spearman, covariance, distance covariance/correlation, x-coverage, distribution distances, raw and standardized slopes, MIC/MAS/MEV/MCN, bin summaries, LOWESS, GAM, and y-SD-normalized measures.

In [7]:
# Copied from D_S2_metrics_computation.ipynb — vectorised correlation metrics.
def vectorised_pearson(x, y):
    xc = x - x.mean(axis=1, keepdims=True)
    yc = y - y.mean(axis=1, keepdims=True)
    num = (xc * yc).sum(axis=1)
    den = np.sqrt((xc**2).sum(axis=1) * (yc**2).sum(axis=1))
    return np.where(den > 0, num / den, np.nan)


def _rank_rows(arr):
    n_cases, n_points = arr.shape
    ranks = np.empty(arr.shape, dtype=np.float64)
    order = arr.argsort(axis=1)
    rows = np.arange(n_cases)[:, None]
    ranks[rows, order] = np.arange(1, n_points + 1, dtype=np.float64)
    return ranks


def vectorised_spearman(x, y):
    return vectorised_pearson(_rank_rows(x), _rank_rows(y))

In [8]:
# Copied from D_S2_metrics_computation.ipynb — shared helpers.
def _to_valid(x, y):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    valid = np.isfinite(x) & np.isfinite(y)
    return x[valid], y[valid]

def _safe_div(a, b):
    if b == 0 or not np.isfinite(b):
        return np.nan
    return a / b

def _minmax01(v):
    v = np.asarray(v, dtype=float)
    lo, hi = np.nanmin(v), np.nanmax(v)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return np.full_like(v, np.nan, dtype=float)
    return (v - lo) / (hi - lo)

def _endpoint_slope(x, y):
    if len(x) < 2: return np.nan
    return _safe_div(float(y[-1] - y[0]), float(x[-1] - x[0]))

def _polyfit_slope(x, y):
    if len(x) < 3 or np.std(x) == 0: return np.nan
    return float(np.polyfit(x, y, 1)[0])

def _segment_masks(n):
    i1, i2 = n // 3, 2 * n // 3
    early = np.zeros(n, bool); early[:i1] = True
    middle = np.zeros(n, bool); middle[i1:i2] = True
    late = np.zeros(n, bool); late[i2:] = True
    return early, middle, late

def _residual_sd(y, y_hat):
    r = y - y_hat
    return float(np.nanstd(r, ddof=1)) if len(r) >= 2 else np.nan

def _r2(y, y_hat):
    ss_res = float(np.nansum((y - y_hat)**2))
    ss_tot = float(np.nansum((y - np.nanmean(y))**2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

def _sign_changes(x_curve, y_curve, tol=1e-6):
    x_curve = np.asarray(x_curve, float)
    y_curve = np.asarray(y_curve, float)
    valid = np.isfinite(x_curve) & np.isfinite(y_curve)
    x_curve, y_curve = x_curve[valid], y_curve[valid]
    if len(x_curve) < 4: return np.nan
    order = np.argsort(x_curve)
    x_curve, y_curve = x_curve[order], y_curve[order]
    dx = np.diff(x_curve)
    dy = np.diff(y_curve)
    ok = dx != 0
    if ok.sum() < 3: return np.nan
    slopes = dy[ok] / dx[ok]
    signs = np.zeros_like(slopes, dtype=int)
    signs[slopes > tol] = 1
    signs[slopes < -tol] = -1
    nz = signs[signs != 0]
    return float(np.sum(nz[1:] != nz[:-1])) if len(nz) >= 2 else 0.0

def _make_bins(x, n_bins=10, bin_type='equal_width'):
    if bin_type == 'equal_width':
        return np.asarray(pd.cut(x, bins=n_bins, labels=False, include_lowest=True, duplicates='drop'), dtype=float)
    elif bin_type == 'equal_count':
        return np.asarray(pd.qcut(x, q=n_bins, labels=False, duplicates='drop'), dtype=float)
    raise ValueError(f'Unknown bin_type: {bin_type}')

In [9]:
# Copied from D_S2_metrics_computation.ipynb — complete per-case metric functions.
def _double_center(a):
    a = a.reshape(-1, 1)
    dist = squareform(pdist(a))
    return dist - dist.mean(axis=0, keepdims=True) - dist.mean(axis=1, keepdims=True) + dist.mean()

def _distance_metrics(x, y):
    r = {'distance_covariance': np.nan, 'distance_correlation': np.nan}
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0: return r
    A, B = _double_center(x), _double_center(y)
    dcov_xy = float(np.sqrt(max((A * B).mean(), 0)))
    dcov_xx = float(np.sqrt(max((A * A).mean(), 0)))
    dcov_yy = float(np.sqrt(max((B * B).mean(), 0)))
    r['distance_covariance'] = dcov_xy
    r['distance_correlation'] = _safe_div(dcov_xy, np.sqrt(dcov_xx * dcov_yy))
    return r

def _mine_metrics(x, y):
    empty = {'MIC': np.nan, 'MAS': np.nan, 'MEV': np.nan, 'MCN': np.nan, 'MIC_minus_r2': np.nan}
    if not HAS_MINEPY or len(x) < 5: return empty
    try:
        mine = MINE(alpha=0.6, c=15)
        mine.compute_score(x, y)
        mic = mine.mic()
        r = pearsonr(x, y)[0] if np.std(x) > 0 and np.std(y) > 0 else np.nan
        return {'MIC': mic, 'MAS': mine.mas(), 'MEV': mine.mev(), 'MCN': mine.mcn(),
                'MIC_minus_r2': mic - r**2 if np.isfinite(r) else np.nan}
    except Exception:
        return empty

def _slope_metrics(x, y, prefix='raw'):
    keys = []
    for method in ['endpoint', 'polyfit']:
        for seg in ['overall', 'early', 'middle', 'late']:
            keys.append(f'{prefix}_{method}_{seg}_slope')
    keys.append(f'{prefix}_segment_strength')
    empty = {k: np.nan for k in keys}
    if len(x) < 6: return empty
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    em, mm, lm = _segment_masks(len(xs))
    segments = {'overall': (xs, ys), 'early': (xs[em], ys[em]),
                'middle': (xs[mm], ys[mm]), 'late': (xs[lm], ys[lm])}
    r = {}
    ep_seg_abs = []
    for seg_name, (sx, sy) in segments.items():
        ep = _endpoint_slope(sx, sy)
        pf = _polyfit_slope(sx, sy)
        r[f'{prefix}_endpoint_{seg_name}_slope'] = ep
        r[f'{prefix}_polyfit_{seg_name}_slope'] = pf
        if seg_name != 'overall':
            ep_seg_abs.append(abs(ep) if np.isfinite(ep) else np.nan)
    r[f'{prefix}_segment_strength'] = float(np.nanmean(ep_seg_abs))
    return r

def _standardized_slope_metrics(x, y):
    xn, yn = _minmax01(x), _minmax01(y)
    if np.any(np.isnan(xn)) or np.any(np.isnan(yn)):
        keys = []
        for method in ['endpoint', 'polyfit']:
            for seg in ['overall', 'early', 'middle', 'late']:
                keys.append(f'standardized_{method}_{seg}_slope')
        keys.append('standardized_segment_strength')
        return {k: np.nan for k in keys}
    return _slope_metrics(xn, yn, prefix='standardized')

def _bin_metrics(x, y, n_bins=10, bin_type='equal_width', min_count=5):
    prefix = f'{bin_type}_bin'
    r = {f'{prefix}_amplitude': np.nan, f'{prefix}_eta_squared': np.nan,
         f'{prefix}_buffer_width_mean': np.nan,
         f'{prefix}_early_buffer_width': np.nan, f'{prefix}_middle_buffer_width': np.nan,
         f'{prefix}_late_buffer_width': np.nan, f'{prefix}_n_valid_bins': 0}
    if len(x) < n_bins: return r
    try: bins = _make_bins(x, n_bins=n_bins, bin_type=bin_type)
    except Exception: return r
    df = pd.DataFrame({'x': x, 'y': y, 'bin': bins}).dropna()
    if df.empty: return r
    bin_stats = []
    for bid, g in df.groupby('bin', observed=True):
        if len(g) < min_count: continue
        yv = g['y'].values
        bw = float(np.nanpercentile(yv, 95) - np.nanpercentile(yv, 5))
        bin_stats.append({'bin': bid, 'x_mean': float(g['x'].mean()),
                          'y_mean': float(yv.mean()), 'buffer_width': bw, 'count': len(g)})
    if len(bin_stats) < 2: return r
    bdf = pd.DataFrame(bin_stats).sort_values('x_mean').reset_index(drop=True)
    r[f'{prefix}_amplitude'] = float(bdf['y_mean'].max() - bdf['y_mean'].min())
    y_global = float(df['y'].mean())
    ss_tot = float(np.sum((df['y'].values - y_global)**2))
    ss_bet = sum(row['count'] * (row['y_mean'] - y_global)**2 for _, row in bdf.iterrows())
    r[f'{prefix}_eta_squared'] = _safe_div(ss_bet, ss_tot)
    r[f'{prefix}_buffer_width_mean'] = float(np.nanmean(bdf['buffer_width']))
    r[f'{prefix}_n_valid_bins'] = len(bdf)
    nv = len(bdf)
    i1, i2 = nv // 3, 2 * nv // 3
    r[f'{prefix}_early_buffer_width'] = float(np.nanmean(bdf.iloc[:i1]['buffer_width']))
    r[f'{prefix}_middle_buffer_width'] = float(np.nanmean(bdf.iloc[i1:i2]['buffer_width']))
    r[f'{prefix}_late_buffer_width'] = float(np.nanmean(bdf.iloc[i2:]['buffer_width']))
    return r

def _x_coverage_metrics(x, n_bins=10):
    r = {'x_bin_count_cv': np.nan, 'x_uniform_ks_distance': np.nan}
    xf = x[np.isfinite(x)]
    if len(xf) < 3: return r
    lo, hi = xf.min(), xf.max()
    if hi > lo:
        xn = np.sort((xf - lo) / (hi - lo))
        n = len(xn)
        r['x_uniform_ks_distance'] = float(max(
            np.max(np.arange(1, n+1) / n - xn), np.max(xn - np.arange(0, n) / n)))
    if len(xf) >= n_bins:
        counts, _ = np.histogram(xf, bins=n_bins)
        mu = counts.mean()
        if mu > 0: r['x_bin_count_cv'] = float(np.std(counts, ddof=1) / mu)
    return r

def _lowess_metrics(x, y, frac=0.25):
    empty = {'lowess_residual_sd': np.nan, 'lowess_curve_amplitude': np.nan,
             'lowess_r2': np.nan, 'lowess_first_derivative_sign_changes': np.nan,
             'lowess_overall_slope': np.nan}
    if len(x) < 5: return empty
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    fitted = lowess(ys, xs, frac=frac, return_sorted=True)
    x_fit, y_fit = fitted[:, 0], fitted[:, 1]
    y_pred = np.interp(xs, x_fit, y_fit)
    return {'lowess_residual_sd': _residual_sd(ys, y_pred),
            'lowess_curve_amplitude': float(np.nanmax(y_fit) - np.nanmin(y_fit)),
            'lowess_r2': _r2(ys, y_pred),
            'lowess_first_derivative_sign_changes': _sign_changes(x_fit, y_fit),
            'lowess_overall_slope': _endpoint_slope(x_fit, y_fit)}

def _gam_metrics(x, y, n_splines=10, lam=0.6, grid_size=200):
    empty = {'gam_residual_sd': np.nan, 'gam_curve_amplitude': np.nan,
             'gam_r2': np.nan, 'gam_first_derivative_sign_changes': np.nan,
             'gam_overall_slope': np.nan}
    if not HAS_PYGAM or len(x) < 10: return empty
    try:
        gam = LinearGAM(gam_s(0, n_splines=n_splines), lam=lam).fit(x.reshape(-1, 1), y)
        x_curve = np.linspace(x.min(), x.max(), grid_size)
        y_curve = gam.predict(x_curve.reshape(-1, 1))
        y_pred = np.interp(x, x_curve, y_curve)
        return {'gam_residual_sd': _residual_sd(y, y_pred),
                'gam_curve_amplitude': float(np.nanmax(y_curve) - np.nanmin(y_curve)),
                'gam_r2': _r2(y, y_pred),
                'gam_first_derivative_sign_changes': _sign_changes(x_curve, y_curve),
                'gam_overall_slope': _endpoint_slope(x_curve, y_curve)}
    except Exception:
        return empty

def _correlation_extra(x, y):
    r = {'covariance': np.nan}
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0: return r
    r['covariance'] = float(np.cov(x, y, ddof=1)[0, 1])
    return r

def _distribution_metrics(x, y, n_bins=10, bin_type='equal_width'):
    prefix = f'{bin_type}_distribution'
    r = {f'{prefix}_ks_distance': np.nan, f'{prefix}_wasserstein_distance': np.nan}
    if len(x) < n_bins: return r
    try: bins = _make_bins(x, n_bins=n_bins, bin_type=bin_type)
    except Exception: return r
    df = pd.DataFrame({'x': x, 'y': y, 'bin': bins}).dropna()
    if df.empty: return r
    valid_bins = np.sort(df['bin'].unique())
    if len(valid_bins) < 3: return r
    nv = len(valid_bins)
    y_low = df[df['bin'].isin(valid_bins[:nv//3])]['y'].values
    y_high = df[df['bin'].isin(valid_bins[2*nv//3:])]['y'].values
    if len(y_low) < 2 or len(y_high) < 2: return r
    r[f'{prefix}_ks_distance'] = float(ks_2samp(y_low, y_high).statistic)
    r[f'{prefix}_wasserstein_distance'] = float(wasserstein_distance(y_low, y_high))
    return r

_Y_SCALE_METRICS = [
    'equal_width_distribution_wasserstein_distance', 'equal_count_distribution_wasserstein_distance',
    'equal_width_bin_amplitude', 'equal_width_bin_buffer_width_mean',
    'equal_width_bin_early_buffer_width', 'equal_width_bin_middle_buffer_width',
    'equal_width_bin_late_buffer_width',
    'equal_count_bin_amplitude', 'equal_count_bin_buffer_width_mean',
    'equal_count_bin_early_buffer_width', 'equal_count_bin_middle_buffer_width',
    'equal_count_bin_late_buffer_width',
    'lowess_residual_sd', 'lowess_curve_amplitude',
    'gam_residual_sd', 'gam_curve_amplitude',
]

def _ysd_normalized(y, metrics):
    y_sd = float(np.nanstd(y, ddof=1))
    r = {'y_sd': y_sd}
    for m in _Y_SCALE_METRICS:
        if m in metrics:
            r[f'{m}_div_y_sd'] = _safe_div(metrics[m], y_sd)
    return r

In [10]:
# Copied from D_S2_metrics_computation.ipynb — one complete metric record per case.
def compute_all_per_case(x, y):
    x, y = _to_valid(x, y)
    m = {}
    m.update(_correlation_extra(x, y))
    m.update(_distance_metrics(x, y))
    m.update(_x_coverage_metrics(x))
    m.update(_distribution_metrics(x, y, bin_type='equal_width'))
    m.update(_distribution_metrics(x, y, bin_type='equal_count'))
    m.update(_slope_metrics(x, y, prefix='raw'))
    m.update(_standardized_slope_metrics(x, y))
    m.update(_mine_metrics(x, y))
    m.update(_bin_metrics(x, y, bin_type='equal_width'))
    m.update(_bin_metrics(x, y, bin_type='equal_count'))
    m.update(_lowess_metrics(x, y))
    m.update(_gam_metrics(x, y))
    m.update(_ysd_normalized(y, m))
    m['n_valid'] = len(x)
    return m

test_started = time.time()
test_metrics = compute_all_per_case(x_all[0].astype(float), y_all[0].astype(float))
test_seconds = time.time() - test_started
assert len(test_metrics) == 74
assert 'MIC' in test_metrics and np.isfinite(test_metrics['MIC'])
print(f'Embedded D_S2 snapshot provides {len(test_metrics)} per-case metrics.')
print(f'Single-case test: MIC={test_metrics["MIC"]:.4f}; {test_seconds:.2f} seconds')

Embedded D_S2 snapshot provides 74 per-case metrics.
Single-case test: MIC=0.1645; 0.04 seconds


## Compute and save resumable metric chunks

Every completed chunk is written immediately. If the full run is interrupted, rerunning the notebook validates existing chunks and continues from the first missing chunk.

In [11]:
def chunk_path(start: int, stop: int) -> Path:
    return CHUNK_DIR / f'metrics_{start:06d}_{stop - 1:06d}.parquet'

def valid_existing_chunk(path: Path, expected_indices: np.ndarray) -> bool:
    if not path.exists() or not RESUME:
        return False
    try:
        schema_names = set(pq.read_schema(path).names)
        if not {'candidate_index', 'case_id', 'pearson_r', 'spearman_rho'}.issubset(schema_names):
            return False
        existing = pd.read_parquet(path, columns=['candidate_index'])
        return np.array_equal(existing['candidate_index'].to_numpy(), expected_indices)
    except Exception:
        return False

run_started = time.time()
computed_chunks = 0
skipped_chunks = 0

for start in tqdm(range(0, n_cases, CHUNK_SIZE), desc='metric chunks'):
    stop = min(start + CHUNK_SIZE, n_cases)
    path = chunk_path(start, stop)
    expected_indices = np.arange(start, stop)
    if valid_existing_chunk(path, expected_indices):
        skipped_chunks += 1
        continue

    x_chunk = x_all[start:stop].astype(np.float64, copy=False)
    y_chunk = y_all[start:stop].astype(np.float64, copy=False)
    pearson_values = vectorised_pearson(x_chunk, y_chunk)
    spearman_values = vectorised_spearman(x_chunk, y_chunk)

    rows = []
    for local_index, candidate_index in enumerate(range(start, stop)):
        metrics = compute_all_per_case(x_chunk[local_index], y_chunk[local_index])
        metrics.update({
            'candidate_index': candidate_index,
            'case_id': cases_df.iloc[candidate_index]['case_id'],
            'pearson_r': float(pearson_values[local_index]),
            'spearman_rho': float(spearman_values[local_index]),
        })
        rows.append(metrics)

    chunk_df = pd.DataFrame(rows)
    metric_names = [
        c for c in chunk_df.columns
        if c not in {'candidate_index', 'case_id', 'pearson_r', 'spearman_rho'}
    ]
    chunk_columns = ['candidate_index', 'case_id', *metric_names, 'pearson_r', 'spearman_rho']
    chunk_df[chunk_columns].to_parquet(path, index=False)
    computed_chunks += 1

metric_files = [chunk_path(start, min(start + CHUNK_SIZE, n_cases)) for start in range(0, n_cases, CHUNK_SIZE)]
missing_chunks = [path for path in metric_files if not path.exists()]
if missing_chunks:
    raise RuntimeError('Metric run is incomplete. Missing chunks: ' + ', '.join(map(str, missing_chunks[:5])))

chunk_metrics_df = pd.concat([pd.read_parquet(path) for path in metric_files], ignore_index=True)
chunk_metrics_df = chunk_metrics_df.sort_values('candidate_index').reset_index(drop=True)
assert len(chunk_metrics_df) == n_cases
assert chunk_metrics_df['candidate_index'].to_numpy().tolist() == list(range(n_cases))
assert chunk_metrics_df['case_id'].tolist() == cases_df['case_id'].tolist()

# Match the original D_S2 final schema exactly: case_id + 74 per-case metrics + Pearson + Spearman.
metric_names = [
    c for c in chunk_metrics_df.columns
    if c not in {'candidate_index', 'case_id', 'pearson_r', 'spearman_rho'}
]
final_columns = ['case_id', *metric_names, 'pearson_r', 'spearman_rho']
metrics_df = chunk_metrics_df[final_columns].copy()

if REFERENCE_METRICS.exists():
    reference_columns = pq.read_schema(REFERENCE_METRICS).names
    assert metrics_df.columns.tolist() == reference_columns, (
        'The regenerated metric schema differs from D/output/S2/metrics_full.parquet.'
    )

assert metrics_df['case_id'].is_unique
assert metrics_df['MIC'].notna().all()
assert len(metrics_df.columns) == 77

final_metrics_path = DATA_DIR / 'metrics_full.parquet'
if final_metrics_path.exists() and not (OVERWRITE_FINAL or RESUME):
    raise FileExistsError(f'{final_metrics_path} exists; set SNR_METRICS_OVERWRITE=1 to replace it.')
metrics_df.to_parquet(final_metrics_path, index=False)

metric_elapsed = time.time() - run_started
metadata = {
    'run_mode': RUN_MODE,
    'metric_definition_source': METRIC_DEFINITION_SOURCE,
    'source_data': str(DATA_DIR),
    'n_cases': int(n_cases),
    'n_points_per_case': int(n_points),
    'n_metric_columns_including_case_id': int(len(metrics_df.columns)),
    'metric_columns': metrics_df.columns.tolist(),
    'mine_settings': {'alpha': 0.6, 'c': 15},
    'has_minepy': HAS_MINEPY,
    'has_pygam': HAS_PYGAM,
    'chunk_size': CHUNK_SIZE,
    'elapsed_seconds_this_run': metric_elapsed,
}
with (DATA_DIR / 'metrics_metadata.json').open('w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2, ensure_ascii=False)

print(f'Computed chunks this run: {computed_chunks}; resumed chunks: {skipped_chunks}')
print(f'Saved {len(metrics_df):,} rows × {len(metrics_df.columns)} columns')
print(f'Metrics: {final_metrics_path}')
print(f'Metadata: {DATA_DIR / "metrics_metadata.json"}')
display(metrics_df.head())

metric chunks: 100%|██████████| 191/191 [1:03:07<00:00, 19.83s/it]


Computed chunks this run: 185; resumed chunks: 6
Saved 95,440 rows × 77 columns
Metrics: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S3_snr_mic_sweep/metrics_full.parquet
Metadata: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/D/Global/output/S3_snr_mic_sweep/metrics_metadata.json


,case_id,covariance,distance_covariance,distance_correlation,x_bin_count_cv,x_uniform_ks_distance,equal_width_distribution_ks_distance,equal_width_distribution_wasserstein_distance,equal_count_distribution_ks_distance,equal_count_distribution_wasserstein_distance,raw_endpoint_overall_slope,raw_polyfit_overall_slope,raw_endpoint_early_slope,raw_polyfit_early_slope,raw_endpoint_middle_slope,raw_polyfit_middle_slope,raw_endpoint_late_slope,raw_polyfit_late_slope,raw_segment_strength,standardized_endpoint_overall_slope,standardized_polyfit_overall_slope,standardized_endpoint_early_slope,standardized_polyfit_early_slope,standardized_endpoint_middle_slope,standardized_polyfit_middle_slope,standardized_endpoint_late_slope,standardized_polyfit_late_slope,standardized_segment_strength,MIC,MAS,MEV,MCN,MIC_minus_r2,equal_width_bin_amplitude,equal_width_bin_eta_squared,equal_width_bin_buffer_width_mean,equal_width_bin_early_buffer_width,equal_width_bin_middle_buffer_width,equal_width_bin_late_buffer_width,equal_width_bin_n_valid_bins,equal_count_bin_amplitude,equal_count_bin_eta_squared,equal_count_bin_buffer_width_mean,equal_count_bin_early_buffer_width,equal_count_bin_middle_buffer_width,equal_count_bin_late_buffer_width,equal_count_bin_n_valid_bins,lowess_residual_sd,lowess_curve_amplitude,lowess_r2,lowess_first_derivative_sign_changes,lowess_overall_slope,gam_residual_sd,gam_curve_amplitude,gam_r2,gam_first_derivative_sign_changes,gam_overall_slope,y_sd,equal_width_distribution_wasserstein_distance_div_y_sd,equal_count_distribution_wasserstein_distance_div_y_sd,equal_width_bin_amplitude_div_y_sd,equal_width_bin_buffer_width_mean_div_y_sd,equal_width_bin_early_buffer_width_div_y_sd,equal_width_bin_middle_buffer_width_div_y_sd,equal_width_bin_late_buffer_width_div_y_sd,equal_count_bin_amplitude_div_y_sd,equal_count_bin_buffer_width_mean_div_y_sd,equal_count_bin_early_buffer_width_div_y_sd,equal_count_bin_middle_buffer_width_div_y_sd,equal_count_bin_late_buffer_width_div_y_sd,lowess_residual_sd_div_y_sd,lowess_curve_amplitude_div_y_sd,gam_residual_sd_div_y_sd,gam_curve_amplitude_div_y_sd,n_valid,pearson_r,spearman_rho
0,S000001,0.053273,0.052296,0.085222,0.093808,0.024789,0.108002,0.405833,0.103333,0.394762,0.411527,0.635432,12.562018,1.714750,7.536156,1.151529,-10.769572,4.173242,10.289249,0.024894,0.038439,0.759913,0.103730,0.455884,0.069659,-0.651483,0.252452,0.622427,0.164492,0.019559,0.164492,5.321928,0.160080,0.980084,0.013787,8.386562,7.532379,8.227291,9.146651,10,1.073497,0.018192,8.503150,7.601718,8.316333,9.319337,10,2.757371,1.001833,0.008894,39.0,0.720377,2.755914,0.810233,0.010153,4.0,0.647352,2.770012,0.146510,0.142513,0.353819,3.027626,2.719259,2.970128,3.302026,0.387542,3.069716,2.744291,3.002273,3.364367,0.995436,0.361671,0.994910,0.292502,500,0.066421,0.059273
1,S000002,0.052361,0.051808,0.085829,0.093808,0.024789,0.108002,0.397078,0.103333,0.385735,0.407910,0.624553,12.164322,1.668865,7.301465,1.123912,-10.410551,4.047621,9.958779,0.025505,0.039050,0.760579,0.104346,0.456527,0.070273,-0.650923,0.253079,0.622676,0.151321,0.013407,0.151321,5.321928,0.146769,0.952179,0.013920,8.114522,7.288044,7.960457,8.849929,10,1.041738,0.018337,8.227344,7.355213,8.046552,9.017036,10,2.667941,0.977908,0.009033,35.0,0.706743,2.666531,0.788379,0.010292,4.0,0.636087,2.680360,0.148144,0.143912,0.355243,3.027400,2.719054,2.969920,3.301768,0.388656,3.069492,2.744113,3.002041,3.364113,0.995366,0.364842,0.994840,0.294132,500,0.067467,0.060280
2,S000003,0.051478,0.051340,0.086465,0.093808,0.024789,0.108002,0.388833,0.103333,0.377233,0.404410,0.614026,11.779524,1.624468,7.074386,1.097190,-10.063174,3.926074,9.639028,0.026136,0.039682,0.761266,0.104983,0.457191,0.070907,-0.650345,0.253727,0.622934,0.144934,0.013527,0.144934,5.321928,0.140235,0.925180,0.014060,7.851305,7.051633,7.702277,8.562830,10,1.011008,0.018490,7.960483,7.116703,7.785520,8.724540,10,2.581411,0.954759,0.009180,35.0,0.693551,2.580047,0.771181,0.010439,4.0,0.625186,2.593619,0.14